## Extração de dados anuais da Steam

### Anos de 2019 e 2020

#### Import da biblioteca pandas:

In [2]:
import pandas as pd

#### Leitura do arquivo de 2019 e 2020

In [3]:
df_2019 = pd.read_csv("dados_mensais_2019.csv")
df_2020 = pd.read_csv("dados_mensais_2020.csv")

# Criação dos dados anuais de 2019

## Limpeza e conversão de tipos

### Remover espaço vazio a esquerda das colunas

In [4]:
df_2019.columns = df_2019.columns.str.lstrip()
df_2020.columns = df_2019.columns.str.lstrip()

### Remoção dos espaços a esquerda dos valores da coluna mês

In [5]:
df_2019['Mês'] = df_2019['Mês'].astype(str).str.lstrip()
df_2020['Mês'] = df_2020['Mês'].astype(str).str.lstrip()

### Conversão de incremento e porcentagem para float

In [6]:
df_2019['Porcentagens'] = (df_2019['Porcentagens'].str.replace('%', '', regex=False).str.strip().astype(float))
df_2019['Incrementos/Decrementos'] = (df_2019['Incrementos/Decrementos'].str.replace('%', '', regex=False).str.strip().astype(float))
df_2020['Porcentagens'] = (df_2020['Porcentagens'].str.replace('%', '', regex=False).str.strip().astype(float))
df_2020['Incrementos/Decrementos'] = (df_2020['Incrementos/Decrementos'].str.replace('%', '', regex=False).str.strip().astype(float))

In [ ]:
print(df_2019.dtypes)
df_2019.head()

Categoria                      str
Especificação                  str
Porcentagens               float64
Incrementos/Decrementos    float64
Mês                            str
Ano                          int64
dtype: object


,Categoria,Especificação,Porcentagens,Incrementos/Decrementos,Mês,Ano
0,OS Version,Windows,95.92,0.06,Janeiro,2019
1,OS Version,Windows 10 64 bit,63.77,-0.02,Janeiro,2019
2,OS Version,Windows 7 64 bit,26.40,0.32,Janeiro,2019
3,OS Version,Windows 8.1 64 bit,3.43,-0.10,Janeiro,2019
4,OS Version,Windows 7,1.52,-0.09,Janeiro,2019


In [7]:
print(df_2020.dtypes)
df_2020.head()

Categoria                      str
Especificação                  str
Porcentagens               float64
Incrementos/Decrementos    float64
Mês                            str
Ano                          int64
dtype: object


,Categoria,Especificação,Porcentagens,Incrementos/Decrementos,Mês,Ano
0,OS Version,Windows,96.09,-0.02,Janeiro,2020
1,OS Version,Windows 10 64 bit,79.23,7.58,Janeiro,2020
2,OS Version,Windows 7 64 bit,13.56,-7.52,Janeiro,2020
3,OS Version,Windows 8.1 64 bit,2.22,0.00,Janeiro,2020
4,OS Version,Windows 7,0.68,-0.06,Janeiro,2020


## Criação dos valores anuais.

Para criar os valores anuais pegaremos os valores de Dezembro como referência e subtrairemos pelos valores da primeira aparição daquela especificação ao longo dos meses daquele ano evitando o erro de peças que estão em dezembro mas não aparecem necessariamente em janeiro.

- Utilização dos valores de dezembro

In [8]:
dezembro_2019 = (
    df_2019[df_2019['Mês'] == 'Dezembro']
    [['Ano', 'Categoria', 'Especificação', 'Porcentagens']]
    .copy()
)

dezembro_2020 = (
    df_2020[df_2020['Mês'] == 'Dezembro']
    [['Ano', 'Categoria', 'Especificação', 'Porcentagens']]
    .copy()
)

In [9]:
print(f"Registros em Dezembro: {len(dezembro_2019)}")
dezembro_2019.head()

Registros em Dezembro: 179


,Ano,Categoria,Especificação,Porcentagens
2100,2019,OS Version,Windows,96.86
2101,2019,OS Version,Windows 10 64 bit,61.09
2102,2019,OS Version,Windows 7 64 bit,33.04
2103,2019,OS Version,Windows 8.1 64 bit,1.79
2104,2019,OS Version,Windows 7,0.60


In [10]:
print(f"Registros em Dezembro: {len(dezembro_2020)}")
dezembro_2020.head()

Registros em Dezembro: 180


,Ano,Categoria,Especificação,Porcentagens
1963,2020,OS Version,Windows,96.51
1964,2020,OS Version,Windows 10 64 bit,91.44
1965,2020,OS Version,Windows 7 64 bit,3.57
1966,2020,OS Version,Windows 8.1 64 bit,1.11
1967,2020,OS Version,Windows 7,0.17


- Primeira aparição

O groupby faz agrupa os dados e pega a primeira linha de cada grupo

In [11]:
primeira_2019 = (df_2019.groupby(['Ano', 'Categoria', 'Especificação'], as_index=False).first()
    [['Ano', 'Categoria', 'Especificação', 'Porcentagens', 'Mês']]
    .rename(columns={'Porcentagens': 'Porcentagem_Inicial','Mês': 'Primeiro_Mês'})
)
primeira_2020 = (df_2020.groupby(['Ano', 'Categoria', 'Especificação'], as_index=False).first()
    [['Ano', 'Categoria', 'Especificação', 'Porcentagens', 'Mês']]
    .rename(columns={'Porcentagens': 'Porcentagem_Inicial','Mês': 'Primeiro_Mês'})
)

- União dos meses

O merge faz a união dos dois dataset a partir das chaves ano, categoria e especificação

In [12]:
resultado_2019 = pd.merge(dezembro_2019,primeira_2019,on=['Ano', 'Categoria', 'Especificação'],how='inner')

resultado_2019['Incrementos/Decrementos'] = (resultado_2019['Porcentagens'] - resultado_2019['Porcentagem_Inicial']).round(2)

resultado_2020 = pd.merge(dezembro_2020,primeira_2020,on=['Ano', 'Categoria', 'Especificação'],how='inner')

resultado_2020['Incrementos/Decrementos'] = (resultado_2020['Porcentagens'] - resultado_2020['Porcentagem_Inicial']).round(2)

- Criação do csv

A variável CSV final serve pra ignorar algumas colunas que foram criadas e deixar no formato dos dados mensais mas sem a coluna mês

In [13]:
csv_final = resultado_2019[[
    'Categoria', 'Especificação', 'Porcentagens', 'Incrementos/Decrementos',
    'Ano'
]]

ARQUIVO_SAIDA_2019 = 'dados_anuais_2019.csv'

csv_final.to_csv(ARQUIVO_SAIDA_2019, index=False)

In [14]:
csv_final = resultado_2020[[
    'Categoria', 'Especificação', 'Porcentagens', 'Incrementos/Decrementos',
    'Ano'
]]

ARQUIVO_SAIDA_2020 = 'dados_anuais_2020.csv'

csv_final.to_csv(ARQUIVO_SAIDA_2020, index=False)